In [242]:
# model implementation
from jax import numpy as jnp
from flax import nnx
import optax

# utilities
from typing import Any
from tqdm.notebook import trange
from functools import partial

# dataloading
import jax_dataloader as jdl
import torchvision
import torch
from torch.utils.data import random_split

In [ ]:
from typing import Callable


dataloader_seed = 42
batch_size = 128
generator = torch.Generator().manual_seed(dataloader_seed)

# By default torchvision gives NCHW but nnx expects NHWC
transform = torchvision.transforms.Compose([torchvision.transforms.ToTensor(), lambda x: torch.einsum("chw->hwc", x)]) 
ds_train_full = torchvision.datasets.CIFAR10(root="/var/tmp", train=True, download=True, transform=transform)
ds_test = torchvision.datasets.CIFAR10(root="/var/tmp", train=False, download=True, transform=transform)

ds_train, ds_val = random_split(ds_train_full, [0.85, 0.15], generator=generator)

dl_train = jdl.DataLoader(ds_train, "pytorch", batch_size=batch_size, shuffle=True, generator=generator)
dl_val = jdl.DataLoader(ds_val, "pytorch", batch_size=batch_size, shuffle=False)
dl_test = jdl.DataLoader(ds_test, "pytorch", batch_size=batch_size, shuffle=False)

/home/roman/repos/learning-ml/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning:

dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)



In [244]:
img, label = next(iter(dl_test))

print(img.shape)

(128, 32, 32, 3)


In [ ]:
class CNN(nnx.Module):
    def __init__(self, rngs: nnx.Rngs, conv_layers = 2, blocks = 1, in_features = 3, internal_features = 32, img_dims = 32):
        assert conv_layers>=blocks
        
        base, remainder = divmod(conv_layers, blocks)  # base=2, remainder=2
        self.blocks = [base + (1 if i < remainder else 0) for i in range(blocks)]
        self.layers = nnx.List([])
        hidden_in, hidden_out = in_features, internal_features
        for block in self.blocks:
            for layer in range(block):
                self.layers.append(
                    nnx.Conv(
                        in_features=hidden_in,
                        out_features=hidden_out,
                        padding="SAME",
                        kernel_size=(3, 3),
                        rngs=rngs,
                    )
                )
                self.layers.append(nnx.relu)
                hidden_in = hidden_out

            self.layers.append(partial(nnx.max_pool, window_shape=(2,2), strides=(2,2)))
            hidden_out = hidden_out * 2

        final_channels = internal_features * (2 ** (blocks - 1))
        final_spatial = img_dims // (2 ** blocks)
        output_size = final_spatial ** 2 * final_channels

        self.fc = nnx.List([nnx.Linear(output_size, 64, rngs=rngs), nnx.Linear(64, 10, rngs=rngs)])

    def __call__(self, X: jnp.ndarray):
        for layer in self.layers:
            X = layer(X)

        X = X.reshape((X.shape[0], -1))

        X = nnx.relu(self.fc[0](X))
        X = self.fc[1](X)

        return X

In [247]:
rngs = nnx.Rngs(42)
model = CNN(rngs=rngs, conv_layers=8, blocks=2)
tx = optax.adam(learning_rate=0.003)
optimiser = nnx.Optimizer(model, tx, wrt=nnx.Param)

class Metrics(nnx.metrics.MultiMetric):
    loss: nnx.metrics.Average
    accuracy: nnx.metrics.Accuracy
metrics = Metrics(loss = nnx.metrics.Average(), accuracy = nnx.metrics.Accuracy())

@nnx.jit
def forward(model, metrics: nnx.Metric, X, y):
    y_hat = model(X)
    error = optax.softmax_cross_entropy_with_integer_labels(y_hat, y).mean()
    metrics.update(values=error, logits=y_hat, labels=y)
    return error

@nnx.jit
def train_step(model: nnx.Module, optimiser: nnx.Optimizer, metrics: nnx.Metric, X: jnp.ndarray, y: jnp.ndarray):
    print("Compiling")
    grads = nnx.grad(forward)(model, metrics, X, y)
    optimiser.update(model, grads)
    return

In [248]:
class Record():
    def __init__(self):
        self.store = {}
        self.has = []

    def add(self, name, train, val):
        if name not in self.has:
            self.store[f"{name}-train-loss"] = []
            self.store[f"{name}-train-acc"] = []
            self.store[f"{name}-val-loss"] = []
            self.store[f"{name}-val-acc"] = []
            self.has.append(name)

        self.store[f"{name}-train-loss"].append(train["loss"])
        self.store[f"{name}-train-acc"].append(train["accuracy"])
        self.store[f"{name}-val-loss"].append(val["loss"])
        self.store[f"{name}-val-acc"].append(val["accuracy"])

record = Record()

In [249]:
epochs = 10

with trange(epochs) as t:
    for i in t:
        model.train()
        metrics.reset()
        for X, y in dl_train:
            train_step(model, optimiser, metrics, X, y)

        t_m = metrics.compute()
        
        model.eval()
        metrics.reset()
        for X, y in dl_val:
            forward(model, metrics, X, y)

        v_m = metrics.compute()

        record.add("test", t_m, v_m)

        t.set_description(f"train l={t_m["loss"]:.2f} a={t_m["accuracy"]:.2f} val l={v_m["loss"]:.2f} a={v_m["accuracy"]:.2f}")


  0%|          | 0/10 [00:00<?, ?it/s]

Compiling
Compiling


In [250]:
import plotly.express as px

px.line(record.store)